# Algorithm 3 — Edit Survival Analysis

**File:** `src/CopilotScope.Collector/Quality/Analyzers.cs` (lines 1–47)

Measures how much of the AI-generated code **survives** after the user's observation window closes.
Two independent telemetry signals are combined into a single survival score.

---

## 1  Input Signals

| Signal | Symbol | Source | Semantics |
|---|---|---|---|
| Four-gram overlap | $F$ | Copilot editor telemetry | Fraction of 4-gram shingles from generated code still present verbatim |
| No-revert | $R$ | Copilot editor telemetry | 1 if the edit was never undone, 0 otherwise (averaged over edits) |

---

## 2  Combined Survival Score

When both signals are present:

$$
\boxed{S = 0.4 \cdot \bar{F} + 0.6 \cdot \bar{R}}
$$

where $\bar{F}$ and $\bar{R}$ are the **per-session means** of each signal over all edits.

When only one signal is available, $S = \bar{F}$ or $S = \bar{R}$ respectively.

**Rationale for $w_R > w_F$:** code can be legitimately *refactored* (low four-gram overlap) while
the semantic intent survives.  Not reverting is the stronger durability signal.

---

## 3  Interpretation Thresholds

| $S$ | Interpretation |
|---|---|
| $S \geq 0.8$ | Code is durable — vast majority survives untouched |
| $0.5 \leq S < 0.8$ | Partial rework — suggestions land but get meaningfully edited |
| $S < 0.5$ | Heavy rewrite — most generated code is replaced or reverted |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def edit_survival(four_gram_samples: list[float], no_revert_samples: list[float]) -> dict:
    fg = np.mean(four_gram_samples) if four_gram_samples else None
    nr = np.mean(no_revert_samples) if no_revert_samples else None

    if fg is not None and nr is not None:
        combined = 0.4 * fg + 0.6 * nr
    elif fg is not None:
        combined = fg
    elif nr is not None:
        combined = nr
    else:
        combined = None

    def interpret(s):
        if s is None: return 'no-data'
        if s >= 0.8: return 'durable'
        if s >= 0.5: return 'partial rework'
        return 'heavy rewrite'

    return {'four_gram': fg, 'no_revert': nr, 'combined': combined,
            'interpretation': interpret(combined)}

# --- Unit results ---
cases = [
    ('Durable — clean acceptance',        [0.90, 0.85, 0.92], [1.0, 1.0, 1.0, 0.9]),
    ('Refactored — low 4-gram, kept',     [0.20, 0.25, 0.30], [0.95, 1.0, 0.9]),
    ('Mixed signals',                     [0.60, 0.55],       [0.70, 0.60, 0.65]),
    ('Mostly reverted',                   [0.30, 0.25],       [0.10, 0.20, 0.15]),
    ('Only 4-gram available',             [0.80, 0.75],       []),
    ('Only no-revert available',          [],                  [0.85, 0.90, 0.80]),
]

print(f'{"Scenario":<38} {"4-gram":>7} {"NoRevert":>9} {"Combined":>9}  Interpretation')
print('-' * 85)
for name, fg, nr in cases:
    r = edit_survival(fg, nr)
    fg_s  = f"{r['four_gram']:.3f}" if r['four_gram']  is not None else '  —'
    nr_s  = f"{r['no_revert']:.3f}" if r['no_revert']  is not None else '  —'
    comb  = f"{r['combined']:.3f}"  if r['combined']   is not None else '  —'
    print(f'{name:<38} {fg_s:>7} {nr_s:>9} {comb:>9}  {r["interpretation"]}')

In [ ]:
# Iso-score contours: combined S as function of (F, R)
F = np.linspace(0, 1, 200)
R = np.linspace(0, 1, 200)
FF, RR = np.meshgrid(F, R)
SS = 0.4 * FF + 0.6 * RR

fig, ax = plt.subplots(figsize=(7, 6))
cs = ax.contourf(FF, RR, SS, levels=20, cmap='RdYlGn')
ct = ax.contour(FF, RR, SS, levels=[0.5, 0.8], colors=['orange', 'green'], linewidths=2)
ax.clabel(ct, fmt={0.5: 'S=0.50', 0.8: 'S=0.80'}, fontsize=11)
plt.colorbar(cs, ax=ax, label='Combined survival score S')
ax.set_xlabel('Four-gram survival $\\bar{F}$')
ax.set_ylabel('No-revert survival $\\bar{R}$')
ax.set_title('Edit Survival: $S = 0.4\\bar{F} + 0.6\\bar{R}$')
plt.tight_layout()
plt.savefig('edit_survival_contour.png', dpi=150)
plt.show()